# Does energy tell Muse Glimmer when it's wrong? — pilot

Scores 200 MMLU questions with Muse Glimmer (Meta's open-weight 30B model) in one forward pass each, and asks:
which confidence score best separates right from wrong answers — softmax confidence, the energy gap (margin),
or the free energy (−logsumexp of the raw logits)? The best one decides which questions a cascade would send to Muse Wonder.

**Runtime → Change runtime type → A100** (or **L4**). A T4 is too small. No API credits are used.

Data: [MMLU](https://huggingface.co/datasets/cais/mmlu) (MIT license), loaded straight from Hugging Face.

In [ ]:
!pip install -q -U "transformers>=5.15" accelerate bitsandbytes datasets scikit-learn

Load the model. The full download is about 60 GB, the slowest step. Precision is picked from the GPU size.

In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL = "meta-models/Muse-Glimmer-30B"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gpu_gb > 20, "Switch the runtime to an L4 or A100"
if gpu_gb > 70:
    quant, mode = None, "bf16"                                              # H100 / A100-80GB
elif gpu_gb > 35:
    quant, mode = BitsAndBytesConfig(load_in_8bit=True), "8-bit"            # A100-40GB
else:
    quant, mode = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                     bnb_4bit_compute_dtype=torch.bfloat16), "4-bit"  # L4
print(f"{torch.cuda.get_device_name(0)} ({gpu_gb:.0f} GB) -> {mode}")

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL, dtype=torch.bfloat16, device_map="auto", quantization_config=quant).eval()

Build the prompt. Glimmer always starts with a reasoning step, so the prompt skips it: it opens the final-answer
channel directly and stops right before the answer letter. The logits at that position are what we score.

In [ ]:
LETTERS = "ABCD"
letter_ids = [tok.encode(l, add_special_tokens=False) for l in LETTERS]
assert all(len(i) == 1 for i in letter_ids), f"letters are not single tokens: {letter_ids}"
letter_ids = [i[0] for i in letter_ids]

def build_prompt(question, choices):
    body = (question + "\n\n" + "\n".join(f"{l}. {c}" for l, c in zip(LETTERS, choices))
            + "\n\nAnswer with the letter of the correct option.")
    text = tok.apply_chat_template([{"role": "user", "content": body}], add_generation_prompt=True,
                                   tokenize=False, reasoning_strength="low")
    if text.endswith("to=self<|message|>"):   # template opened the reasoning channel: close it empty
        text += "<|eom|><|start|>assistant"
    return text + "<|message|>The answer is ("

print(build_prompt("What is 2 + 2?", ["3", "4", "5", "6"])[-300:])

In [ ]:
from datasets import load_dataset
N = 200
ds = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=0).select(range(N))

Score every question: one forward pass, raw logits over the whole vocabulary at the answer position.

In [ ]:
import pandas as pd

rows, t0 = [], time.time()
for ex in ds:
    ids = tok(build_prompt(ex["question"], ex["choices"]), return_tensors="pt",
              add_special_tokens=False).input_ids.to(model.device)
    with torch.no_grad():
        z = model(input_ids=ids).logits[0, -1].float()     # raw logits, full vocabulary
    zl = z[letter_ids]
    top2 = zl.topk(2).values
    pred = int(zl.argmax())
    rows.append(dict(
        subject=ex["subject"], label=ex["answer"], pred=pred, correct=pred == ex["answer"],
        msp=float(zl.softmax(0).max()),                      # softmax confidence over A-D
        margin=float(top2[0] - top2[1]),                     # energy gap: survives in API log-probs
        neg_free_energy=float(torch.logsumexp(z, 0)),        # needs raw logits: lost after log-softmax
        neg_free_energy_letters=float(torch.logsumexp(zl, 0)),
        letter_mass=float(z.softmax(0)[letter_ids].sum()),   # sanity check: probability landing on A-D
    ))
print(f"{len(rows)} questions in {time.time() - t0:.0f}s")
df = pd.DataFrame(rows)
df.to_csv("glimmer_mmlu_pilot.csv", index=False)

**Reading the results.** AUROC is how well a score separates right from wrong answers (0.5 = chance, 1 = perfect).
With 200 questions, differences under about 0.05 are noise. If `letter_mass` is well below 0.9, the model isn't
answering with a letter at that position and the prompt needs fixing before anything else.
Glimmer soft-caps its logits at ±20, which compresses the free energy; keep that in mind if it underperforms.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

print(f"accuracy {df.correct.mean():.3f} | mean probability on A-D {df.letter_mass.mean():.3f}")
scores = ["msp", "margin", "neg_free_energy", "neg_free_energy_letters"]
print("\nAUROC, right vs wrong:")
for s in scores:
    print(f"  {s:25s} {roc_auc_score(df.correct, df[s]):.3f}")

# Cascade view: Glimmer keeps its most confident questions, the rest would go to Muse Wonder.
coverage = np.linspace(0.1, 1, 10)
for s in scores:
    ranked = df.sort_values(s, ascending=False).correct.values
    plt.plot(coverage, [ranked[: max(1, int(c * len(ranked)))].mean() for c in coverage], marker="o", label=s)
plt.xlabel("share of questions Glimmer keeps"); plt.ylabel("Glimmer accuracy on kept questions")
plt.legend(); plt.grid(alpha=0.3); plt.show()